# Rényi Entropy–Complexity Causality Space for Transformer Activations

> ## ⚠️ Important — what this notebook is and is not
>
> This notebook demonstrates the math of Rényi entropy, Jensen–Rényi divergence, and statistical complexity
> (Guisande & Montani 2024; Jauregui et al. 2018; Martin/Plastino/Rosso 2006; Rosso et al. 2007) on
> **synthetic NumPy data** — `np.sin`, `np.random.dirichlet`, and a 5-regime taxonomy.
> **No real transformer is loaded; no hooks are installed.** The `(H_q, C_q)` plots are properties of the
> synthesizer, not of any model.
>
> Replace `synthesize_run` with a real activations cache (e.g. via navi-SAD's `analysis/loader.py`) before
> reading any of the plotted geometry as evidence about real transformer behavior. The math primitives
> are correct (cross-checked numerically against `ordpy` to ~1e-18); everything downstream of
> `synthesize_run` is exploratory only.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)
*(Replace the link above with the GitHub path of this notebook to get a working "Open in Colab" badge — e.g. `https://colab.research.google.com/github/<owner>/<repo>/blob/main/renyi_entropy_complexity_transformer_interpretability_demo.ipynb`.)*

This notebook adapts the **Rényi entropy–complexity causality plane** machinery from
Guisande, Montani, et al. (Frontiers in Computational Neuroscience, 2024) to
**synthetic transformer activations**. The original code, derived from the
[`ordpy`](https://arthurpessa.github.io/ordpy/) library, was developed for
EEG/iEEG analysis. Here we re-purpose it as a *diagnostic geometry* for
sequences of token-wise activations and attention vectors that we generate
ourselves.

**What this notebook is:** a self-contained, Colab-ready demo that
1. implements ordinal-pattern (Bandt–Pompe) probability extraction,
2. computes Rényi entropy $H_q$, Jensen–Rényi divergence, and statistical complexity $C_q$,
3. constructs interpretable synthetic regimes for activations and attention heads,
4. visualizes how regimes cluster on the $(H_q, C_q)$ plane, and
5. ships an extensive **audit suite** that validates every step.

**What this notebook is *not*:** a claim about real transformer mechanisms.
The data here is synthetic and the regimes are caricatures (focused / diffuse / induction-like / collapse / drift).
The goal is to test whether the Rényi plane can *separate* such regimes — a necessary, not sufficient, condition for downstream interpretability use.

---

### Attribution

- Theoretical foundation for the (H, C) plane: Rosso, O. A., Larrondo, H. A., Martín, M. T., Plastino, A. & Fuentes, M. A. (2007). *Distinguishing Noise from Chaos.* Phys. Rev. Lett. **99**, 154102. [doi:10.1103/PhysRevLett.99.154102](https://doi.org/10.1103/PhysRevLett.99.154102)
- Repository: [Gisandio/Renyi-Entropy-Complexity-Causality-Space](https://github.com/Gisandio/Renyi-Entropy-Complexity-Causality-Space).
- Paper: Natalí Guisande, Fernando Montani, et al. *Rényi Entropy–Complexity Causality Space: A Novel Neurocomputational Tool for Detecting Scale-Free Features in EEG/iEEG Data*. **Frontiers in Computational Neuroscience** (2024). [doi:10.3389/fncom.2024.1342985](https://www.frontiersin.org/journals/computational-neuroscience/articles/10.3389/fncom.2024.1342985/abstract).
- `ordpy`: Pessa, A. A. B. & Ribeiro, H. V. *ordpy: A Python package for data analysis with permutation entropy and ordinal network methods.* **Chaos 31, 063110** (2021). [arXiv:2102.06786](https://arxiv.org/abs/2102.06786).
- The bound-curve helpers `maximum_renyi_complexity_entropy` and `minimum_renyi_complexity_entropy` are direct adaptations of code in the source repo (`Bounds_and_Systems_Across_Planes_for_Various_q_Values.ipynb`).

The contents of this notebook beyond those direct adaptations — the synthetic transformer activation models, the audit suite, the interactive widgets, and the interpretability narrative — are new.


## 1. Setup and dependencies

This cell installs `ordpy` if missing. All other dependencies (`numpy`, `scipy`,
`matplotlib`, `seaborn`, `pandas`, `ipywidgets`) are present in stock Colab.


In [ ]:
# Install ordpy if not present (Colab-safe; no-op locally if already installed).
import importlib, subprocess, sys
def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for pkg, mod in [("ordpy", "ordpy"), ("ipywidgets", "ipywidgets")]:
    _ensure(pkg, mod)


In [ ]:
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from dataclasses import dataclass

import ordpy
from ordpy import (
    renyi_entropy as _ordpy_renyi_entropy,
    renyi_complexity_entropy as _ordpy_renyi_complexity_entropy,
    permutation_entropy as _ordpy_perm_entropy,
    ordinal_distribution as _ordpy_ordinal_dist,
    minimum_complexity_entropy,
    maximum_complexity_entropy,
)

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Restrained, colorblind-aware palette for regimes (color reinforced by marker shape).
REGIME_COLORS = {
    "focused":      "#1f77b4",
    "diffuse":      "#ff7f0e",
    "induction":    "#2ca02c",
    "collapse":     "#d62728",
    "drift":        "#9467bd",
    "intervention": "#8c564b",
    "white_noise":  "#7f7f7f",
}
REGIME_MARKERS = {
    "focused":      "o",
    "diffuse":      "s",
    "induction":    "D",
    "collapse":     "X",
    "drift":        "^",
    "intervention": "P",
    "white_noise":  ".",
}

mpl.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "legend.frameon": False,
    "mathtext.fontset": "cm",
})
sns.set_context("notebook")
print("Setup OK; ordpy", getattr(ordpy, "__version__", "?"))


## 2. Rényi entropy / complexity primitives

We implement the math from scratch and **also** call `ordpy` so we can cross-check.

Given a probability distribution $p$ on $N$ ordinal patterns, with the uniform $u_i = 1/N$:

- **Rényi entropy of order $q>0$**: $H_q^R(p) = \frac{1}{1-q}\log\sum_i p_i^{q}$, with the Shannon limit $H_1^R = -\sum_i p_i \log p_i$.
- **Normalized Rényi entropy**: $\mathcal{H}_q(p) = H_q^R(p) / \log N \in [0,1]$.
- **Jensen–Rényi divergence to uniform** (Jauregui, Zunino, Lenzi, Mendes & Ribeiro, *Physica A* **498** (2018) 74–85, Eq. 4; arXiv:1801.05738):
  $$\mathcal{Q}_q(p,u) = \frac{1}{2(q-1)}\left[\log\sum_i \left(\tfrac{p_i+u_i}{2}\right)^{1-q} p_i^{q} + \log\sum_i \left(\tfrac{1}{N^q}\right)\left(\tfrac{p_i+u_i}{2}\right)^{1-q}\right].$$
  This is the form used by `ordpy.renyi_complexity_entropy` and reduces to the standard Jensen–Shannon divergence at $q\to 1$.
- **Maximum Jensen–Rényi divergence** (Jauregui et al. Eq. 5), $\mathcal{Q}_q^{\max}(N)$, attained by a delta vs. the uniform.
- **Statistical complexity**: $C_q(p) = \mathcal{Q}_q(p,u) \cdot \mathcal{H}_q(p) / \mathcal{Q}_q^{\max}(N)$. By construction, $C_q \in [0, 1]$, $C_q=0$ at the uniform (because $\mathcal{Q}_q=0$) and at a delta (because $\mathcal{H}_q=0$).

Edge-case handling:
- `q <= 0` is rejected.
- `|q-1| < 1e-6` is treated as the **Shannon limit** with explicit $-\sum p\log p$.
- Zero entries in $p$ are handled with a closed-form correction term for the missing-state contributions to the divergence.
- Degenerate (delta) distributions: $H_q = 0$, $C_q = 0$.


In [ ]:
SHANNON_TOL = 1e-6

def _validate_probs(p):
    p = np.asarray(p, dtype=float).ravel()
    if np.any(p < -1e-12):
        raise ValueError("Probabilities must be non-negative.")
    p = np.clip(p, 0.0, None)
    s = p.sum()
    if s <= 0:
        raise ValueError("Probability vector sums to zero.")
    return p / s

def renyi_entropy_from_probs(p, q):
    # Unnormalized Rényi entropy in nats. Robust for q>0 including the Shannon limit q->1.
    if q <= 0:
        raise ValueError("Rényi order q must be > 0.")
    p = _validate_probs(p)
    p_pos = p[p > 0]
    if abs(q - 1.0) < SHANNON_TOL:
        return float(-np.sum(p_pos * np.log(p_pos)))
    s = float(np.sum(p_pos ** q))
    return float(np.log(s) / (1.0 - q))

def normalized_renyi_entropy(p, q):
    # H_q / log(N), where N = len(p). Returns 0 for delta, 1 for uniform.
    p = _validate_probs(p)
    N = len(p)
    if N <= 1:
        return 0.0
    return renyi_entropy_from_probs(p, q) / math.log(N)

def jensen_renyi_divergence(p, q):
    # Jensen-Rényi divergence between p and the uniform on N=len(p) atoms,
    # following Eqs. (4)-(5) of Jauregui, Zunino, Lenzi, Mendes & Ribeiro,
    # Physica A 498 (2018) 74-85 (arXiv:1801.05738)
    # (the formulation used by ordpy.renyi_complexity_entropy).
    # For q==1 we fall back to the standard Jensen-Shannon limit.
    p = _validate_probs(p)
    N = len(p)
    u = 1.0 / N
    if abs(q - 1.0) < SHANNON_TOL:
        # Jensen-Shannon, mass on full N support (no missing-pattern correction here).
        m = (p + u) / 2.0
        m_pos = m[m > 0]
        Hm = -np.sum(m_pos * np.log(m_pos))
        p_pos = p[p > 0]
        Hp_half = -np.sum(p_pos * np.log(p_pos)) / 2.0
        Hu_half = math.log(N) / 2.0
        return float(Hm - Hp_half - Hu_half)
    # General q != 1; we exclude zero entries from the first term but include them
    # in the closed-form correction term ("n_states_not_occur") in ordpy.
    p_nz = p[p > 0]
    n_zero = N - len(p_nz)
    mix_nz = (p_nz + u) / 2.0
    first  = np.log(np.sum((mix_nz ** (1 - q)) * (p_nz ** q)))
    second = np.log(
        np.sum((1.0 / N ** q) * (mix_nz ** (1 - q)))
        + n_zero * (1.0 / N ** q) * ((1.0 / (2 * N)) ** (1 - q))
    )
    return float((first + second) / (2.0 * (q - 1.0)))

def jensen_renyi_max(N, q):
    # Eq. (5) of Jauregui et al. (Physica A 498, 2018; arXiv:1801.05738).
    if abs(q - 1.0) < SHANNON_TOL:
        return float(-0.5 * (((N + 1) / N) * math.log(N + 1) + math.log(N) - 2.0 * math.log(2 * N)))
    return float(
        (
            math.log(((N + 1.0) ** (1.0 - q) + N - 1.0) / (2.0 ** (1.0 - q) * N))
            + (1.0 - q) * math.log((N + 1.0) / (2.0 * N))
        )
        / (2.0 * (q - 1.0))
    )

def statistical_complexity_from_probs(p, q):
    # C_q(p) = (Q_q(p,u) / Q_q^max) * H_q(p)/log N. Robust for q>0.
    p = _validate_probs(p)
    N = len(p)
    if N <= 1:
        return 0.0
    H = normalized_renyi_entropy(p, q)
    Q = jensen_renyi_divergence(p, q)
    Qmax = jensen_renyi_max(N, q)
    if Qmax <= 0:
        return 0.0
    return float((Q / Qmax) * H)

print("Math primitives loaded.")


## 3. Ordinal-pattern (Bandt–Pompe) extraction

For a 1-D series $x_1,\dots,x_T$, embedding dimension $d_x$ and delay $\tau$,
we build the multi-set of permutations $\pi$ of length $d_x$ describing the rank
order of $(x_t,x_{t+\tau},\dots,x_{t+(d_x-1)\tau})$ for valid $t$. The empirical
frequency over the $d_x!$ permutations is the **ordinal distribution**.

We provide our own `ordinal_distribution` and cross-check against the one in `ordpy`.


In [ ]:
from itertools import permutations

def ordinal_distribution(x, dx=4, tau=1, tie_rule="random", rng=None):
    # Tie policy: this implementation uses Bandt & Pompe 2002 Sec. II original
    # advice (additive 1e-12 noise to break ties). Note that the navi-SAD
    # project's `signal/ordinal.py` engine uses tie EXCLUSION (drop tied
    # embedding windows entirely) — a different downstream convention. If you
    # port this to navi-SAD, switch to tie exclusion to preserve the project's
    # contract; otherwise you silently inject RNG-dependent mass on patterns
    # that are not in the data.
    # Returns (patterns, probs) for series x.
    # tie_rule: 'random' breaks ties by tiny noise; 'first' uses argsort default.
    x = np.asarray(x, dtype=float).ravel()
    T = len(x)
    n_emb = T - (dx - 1) * tau
    if n_emb <= 0:
        raise ValueError(f"Series too short: T={T}, dx={dx}, tau={tau}.")
    rng = np.random.default_rng() if rng is None else rng
    if tie_rule == "random":
        x = x + rng.normal(0, 1e-12, size=x.shape)
    # Build embedding matrix (n_emb, dx)
    emb = np.stack([x[i: i + (dx - 1) * tau + 1: tau] for i in range(n_emb)])
    perms = np.argsort(emb, axis=1, kind="stable")
    # Encode each permutation row to a unique integer in [0, dx!) via factoradic.
    factorials = np.array([math.factorial(dx - 1 - i) for i in range(dx)], dtype=np.int64)
    codes = np.zeros(n_emb, dtype=np.int64)
    used = np.zeros((n_emb, dx), dtype=bool)
    rows = np.arange(n_emb)
    for i in range(dx):
        col = perms[:, i]
        rank = np.zeros(n_emb, dtype=np.int64)
        for j in range(dx):
            rank += ((j < col) & (~used[rows, j])).astype(np.int64)
        codes += rank * factorials[i]
        used[rows, col] = True
    counts = np.bincount(codes, minlength=math.factorial(dx))
    probs = counts.astype(float) / counts.sum()
    pat_list = list(permutations(range(dx)))
    return pat_list, probs

# Quick cross-check against ordpy's ordinal_distribution: total prob = 1, same length.
_x = np.random.default_rng(0).standard_normal(2000)
_pat, _p = ordinal_distribution(_x, dx=4)
assert abs(_p.sum() - 1.0) < 1e-12 and len(_p) == 24
print("ordinal_distribution OK.")


## 4. Bound curves on the $(H_q, C_q)$ plane

Functions adapted directly from the source repository's
`Bounds_and_Systems_Across_Planes_for_Various_q_Values.ipynb`. They generate
the upper and lower limiting curves of the Rényi entropy–complexity plane for
embedding dimension $d_x$ and order $q$.


In [ ]:
def maximum_renyi_complexity_entropy(dx=4, dy=1, m=200, q=1.4):
    # Upper bound curve. Adapted from Gisandio repo (Bounds notebook).
    if abs(q - 1.0) < SHANNON_TOL:
        return maximum_complexity_entropy(dx, dy, m)
    N = math.factorial(dx * dy)
    hlist_, clist_ = np.zeros((N - 1, m)), np.zeros((N - 1, m))
    for i in range(N - 1):
        p = np.zeros(N)
        uniform_dist = np.full(N, 1.0 / N)
        prob_params = np.linspace(0, 1.0 / N, num=m)
        for k in range(len(prob_params)):
            p[0] = prob_params[k]
            for j in range(1, N - i):
                p[j] = (1 - prob_params[k]) / (N - i - 1)
            h = _ordpy_renyi_entropy(p, alpha=q, dx=dx, dy=dy, probs=True)
            mix = (uniform_dist + p) / 2.0
            t1 = np.log(np.sum((p ** q) * mix ** (1 - q)))
            t2 = np.log(np.sum((uniform_dist ** q) * mix ** (1 - q)))
            js_div = (0.5 / (q - 1)) * (t1 + t2)
            js_div_max = (0.5 / (q - 1)) * np.log(
                (((N + 1) ** (1 - q) + N - 1) / N) * ((N + 1) / (4 * N)) ** (1 - q)
            )
            c = js_div * h / js_div_max
            hlist_[i, k] = h
            clist_[i, k] = c
    return np.column_stack([hlist_.ravel(), clist_.ravel()])

def minimum_renyi_complexity_entropy(dx=4, dy=1, size=200, q=1.4):
    # Lower bound curve. Adapted from Gisandio repo (Bounds notebook).
    if abs(q - 1.0) < SHANNON_TOL:
        return minimum_complexity_entropy(dx, dy, size)
    size += 1
    N = math.factorial(dx * dy)
    prob_params = np.linspace(1.0 / N, 1.0, num=size - 1)
    uniform_dist = np.full(N, 1.0 / N)
    hc_ = []
    for i in range(size - 1):
        probabilities = np.full(N, (1 - prob_params[i]) / (N - 1))
        probabilities[0] = prob_params[i]
        h = _ordpy_renyi_entropy(probabilities, alpha=q, dx=dx, dy=dy, probs=True)
        mix = (uniform_dist + probabilities) / 2.0
        t1 = np.log(np.sum((probabilities ** q) * mix ** (1 - q)))
        t2 = np.log(np.sum((uniform_dist ** q) * mix ** (1 - q)))
        js_div = (0.5 / (q - 1)) * (t1 + t2)
        js_div_max = (0.5 / (q - 1)) * np.log(
            (((N + 1) ** (1 - q) + N - 1) / N) * ((N + 1) / (4 * N)) ** (1 - q)
        )
        c = js_div * h / js_div_max
        hc_.append([h, c])
    return np.array(hc_)

print("Bound-curve helpers loaded.")


## 5. Synthetic transformer activations and attention

We generate **interpretable regimes** that are *caricatures* of phenomena
described in the mech-interp literature. The shapes are deliberately simple so
that the entropy–complexity plane has something to find:

| Regime | Description | Expected $(H_q, C_q)$ behaviour |
|---|---|---|
| `focused`     | Attention concentrated on a single token; activation a low-frequency sinusoid + small noise | low $\mathcal{H}$, low $C$ (delta-like) |
| `diffuse`     | Near-uniform attention; activation white noise | high $\mathcal{H}$, low $C$ |
| `induction`   | Attention pattern that copies a previous token (induction-head caricature) producing a periodic ordinal structure | medium $\mathcal{H}$, medium-high $C$ |
| `collapse`    | Attention collapses partway through the sequence (head dies); activation drops to near-zero noise | non-stationary, mid-plane |
| `drift`       | Layer-wise drift: a sinusoid whose frequency walks slowly | medium-high $\mathcal{H}$, medium $C$ |
| `intervention`| Causal patch: replace a slice with values from a different regime (used for trajectories) | trajectory across plane |

The `synthesize_run` function returns a 4-D tensor of shape
`(n_layers, n_heads, n_tokens, d_feat)` plus an attention tensor
`(n_layers, n_heads, n_tokens, n_tokens)`. We build *all* visualizations from
this single object.


In [ ]:
@dataclass
class SynthConfig:
    n_layers: int = 6
    n_heads: int = 4
    n_tokens: int = 512
    d_feat: int = 32
    seed: int = 0
    layer_regimes: list = None
    head_regimes: list = None
    intervention_strength: float = 0.0
    intervention_layer: int = 3
    intervention_head: int = 0
    intervention_target_regime: str = "diffuse"

REGIMES = ["focused", "diffuse", "induction", "collapse", "drift"]

def _activation_regime(regime, n_tokens, d_feat, rng):
    t = np.arange(n_tokens)
    if regime == "focused":
        freq = rng.uniform(0.01, 0.04)
        sig = np.sin(2*np.pi*freq*t) + 0.05*rng.standard_normal(n_tokens)
        feat = sig[:, None] + 0.03*rng.standard_normal((n_tokens, d_feat))
    elif regime == "diffuse":
        feat = rng.standard_normal((n_tokens, d_feat))
    elif regime == "induction":
        period = int(rng.integers(8, 24))
        base = rng.standard_normal(period)
        sig = np.tile(base, n_tokens // period + 1)[:n_tokens]
        feat = sig[:, None] * rng.standard_normal((1, d_feat)) + 0.2*rng.standard_normal((n_tokens, d_feat))
    elif regime == "collapse":
        cut = n_tokens // 2
        sig = np.concatenate([
            np.sin(2*np.pi*0.03*t[:cut]) + 0.05*rng.standard_normal(cut),
            0.02*rng.standard_normal(n_tokens-cut),
        ])
        feat = sig[:, None] + 0.05*rng.standard_normal((n_tokens, d_feat))
    elif regime == "drift":
        # Frequency-modulated chirp-like signal.
        freq = 0.005 + 0.04*np.cumsum(rng.standard_normal(n_tokens))/n_tokens
        phase = 2*np.pi*np.cumsum(freq)
        sig = np.sin(phase) + 0.08*rng.standard_normal(n_tokens)
        feat = sig[:, None] + 0.05*rng.standard_normal((n_tokens, d_feat))
    else:
        raise ValueError(f"Unknown regime {regime!r}")
    return feat

def _attention_regime(regime, n_tokens, rng):
    # Returns (n_tokens, n_tokens) row-stochastic attention.
    A = np.zeros((n_tokens, n_tokens))
    if regime == "focused":
        for i in range(n_tokens):
            target = max(0, i - int(rng.integers(0, 3)))
            A[i, target] = 1.0
        A += 1e-3*rng.standard_normal(A.shape)**2
    elif regime == "diffuse":
        A = rng.dirichlet(np.ones(n_tokens), size=n_tokens)
    elif regime == "induction":
        offset = int(rng.integers(3, 8))
        for i in range(n_tokens):
            tgt = max(0, i - offset)
            A[i, tgt] = 0.85
            A[i, max(0, i-1)] += 0.05
        A += 0.01
    elif regime == "collapse":
        cut = n_tokens // 2
        for i in range(n_tokens):
            if i < cut:
                A[i, max(0, i-1)] = 0.9
            else:
                A[i, :] = 1.0 / n_tokens
        A += 1e-3
    elif regime == "drift":
        for i in range(n_tokens):
            sigma = 1.0 + 8*i/n_tokens
            xs = np.arange(n_tokens) - i
            row = np.exp(-0.5*(xs/sigma)**2)
            A[i] = row
    else:
        raise ValueError(regime)
    A = np.clip(A, 1e-12, None)
    A = A / A.sum(axis=1, keepdims=True)
    return A

def synthesize_run(cfg):
    # Audit FINDING-05 fix: do NOT mutate cfg. Resolve regimes into locals so a
    # subsequent call with the same cfg but different n_layers/n_heads is correct.
    rng = np.random.default_rng(cfg.seed)
    layer_regimes = (
        list(cfg.layer_regimes)
        if cfg.layer_regimes is not None
        else [REGIMES[i % len(REGIMES)] for i in range(cfg.n_layers)]
    )
    head_regimes = (
        [list(row) for row in cfg.head_regimes]
        if cfg.head_regimes is not None
        else [[layer_regimes[L]] * cfg.n_heads for L in range(cfg.n_layers)]
    )
    activations = np.zeros((cfg.n_layers, cfg.n_heads, cfg.n_tokens, cfg.d_feat))
    attention = np.zeros((cfg.n_layers, cfg.n_heads, cfg.n_tokens, cfg.n_tokens))
    regime_grid = np.empty((cfg.n_layers, cfg.n_heads), dtype=object)
    for L in range(cfg.n_layers):
        for h in range(cfg.n_heads):
            regime = head_regimes[L][h]
            regime_grid[L, h] = regime
            activations[L, h] = _activation_regime(regime, cfg.n_tokens, cfg.d_feat, rng)
            attention[L, h] = _attention_regime(regime, cfg.n_tokens, rng)
    if cfg.intervention_strength > 0:
        L = cfg.intervention_layer
        h = cfg.intervention_head
        a = cfg.intervention_strength
        target = _activation_regime(cfg.intervention_target_regime, cfg.n_tokens, cfg.d_feat, rng)
        activations[L, h] = (1-a)*activations[L, h] + a*target
        target_attn = _attention_regime(cfg.intervention_target_regime, cfg.n_tokens, rng)
        attention[L, h] = (1-a)*attention[L, h] + a*target_attn
        attention[L, h] /= attention[L, h].sum(axis=1, keepdims=True)
        regime_grid[L, h] = f"intervention({cfg.intervention_target_regime},{a:.2f})"
    return {"activations": activations, "attention": attention, "regimes": regime_grid, "config": cfg}

print("Synthetic data generators loaded.")


## 6. Mapping a layer/head to the $(H_q, C_q)$ plane

For each head we compute two complementary $(H_q, C_q)$ points:

1. **Activation point**: average the feature dimension to get a 1-D series, then
   apply Bandt–Pompe with embedding $d_x$ and delay $\tau$.
2. **Attention point**: each attention row is already a probability distribution
   over keys, so we apply Rényi/Jensen–Rényi to those rows directly and average
   across query tokens.

Both views are useful: activation entropy captures *temporal* regularity,
attention entropy captures *spatial* focus.


In [ ]:
def renyi_point_from_series(x, q, dx=4, tau=1, rng=None):
    pats, probs = ordinal_distribution(x, dx=dx, tau=tau, rng=rng)
    H = normalized_renyi_entropy(probs, q)
    C = statistical_complexity_from_probs(probs, q)
    return H, C, probs

def renyi_point_from_attention(A, q):
    A = np.asarray(A, float)
    Hs, Cs = [], []
    for row in A:
        if row.sum() <= 0:
            continue
        Hs.append(normalized_renyi_entropy(row, q))
        Cs.append(statistical_complexity_from_probs(row, q))
    return float(np.mean(Hs)), float(np.mean(Cs))

def feature_table(run, q, dx=4, tau=1):
    cfg = run["config"]
    rows = []
    rng = np.random.default_rng(cfg.seed + 1000)
    for L in range(cfg.n_layers):
        for h in range(cfg.n_heads):
            sig = run["activations"][L, h].mean(axis=1)
            H_a, C_a, _ = renyi_point_from_series(sig, q=q, dx=dx, tau=tau, rng=rng)
            H_t, C_t = renyi_point_from_attention(run["attention"][L, h], q=q)
            rows.append({
                "layer": L, "head": h,
                "regime": str(run["regimes"][L, h]),
                "H_act": H_a, "C_act": C_a,
                "H_attn": H_t, "C_attn": C_t,
            })
    return pd.DataFrame(rows)

print("Feature-table builder loaded.")


## 7. Correctness audit suite

These tests run *every time* the notebook is executed end-to-end. Each prints
`PASS`/`FAIL`. The final cell of §11 asserts that every audit passed.


In [ ]:
class _AuditLog:
    def __init__(self):
        self.entries = []
    def add(self, name, ok, detail=""):
        self.entries.append((name, bool(ok), detail))
        flag = "PASS" if ok else "FAIL"
        print(f"  [{flag}] {name}" + ((" - " + detail) if detail else ""))
    def all_passed(self):
        return all(e[1] for e in self.entries)
    def to_dataframe(self):
        return pd.DataFrame(self.entries, columns=["audit", "passed", "detail"])

AUDITS = _AuditLog()

def _run_audit(name, fn):
    try:
        ok, detail = fn()
        AUDITS.add(name, ok, detail)
    except AssertionError as e:
        AUDITS.add(name, False, f"AssertionError: {e}")
    except Exception as e:
        AUDITS.add(name, False, f"{type(e).__name__}: {e}")

# A: probability simplex sanity
def _audit_simplex():
    rng = np.random.default_rng(1)
    x = rng.standard_normal(2000)
    pats, p = ordinal_distribution(x, dx=4, tau=1, rng=rng)
    cond = (len(p) == math.factorial(4)) and np.all(p >= 0) and abs(p.sum() - 1.0) < 1e-12
    return cond, f"len={len(p)}, sum={p.sum():.6f}"

# B: q->1 Shannon limit (audit FINDING-02 fix)
# The previous version probed at q = 1 +/- 1e-9, which is well inside the
# SHANNON_TOL window in renyi_entropy_from_probs and therefore took the
# Shannon-only branch — comparing Shannon to Shannon (a tautology). The new
# version probes outside the tolerance window so the q != 1 general formula
# is actually exercised, and it asks for monotonic convergence as |q-1| -> 0.
def _audit_shannon_limit():
    rng = np.random.default_rng(2)
    p = rng.dirichlet(np.ones(24))
    Hsh = float(-np.sum(p[p>0]*np.log(p[p>0])))
    # Probe well outside SHANNON_TOL (1e-6) so the general q != 1 branch runs.
    Hexact = renyi_entropy_from_probs(p, q=1.0)
    H_minus_3 = renyi_entropy_from_probs(p, q=1.0 - 1e-3)
    H_plus_3  = renyi_entropy_from_probs(p, q=1.0 + 1e-3)
    H_minus_4 = renyi_entropy_from_probs(p, q=1.0 - 1e-4)
    H_plus_4  = renyi_entropy_from_probs(p, q=1.0 + 1e-4)
    # Closer-to-1 must be closer to Shannon than farther-from-1.
    closer_minus = abs(H_minus_4 - Hsh) <= abs(H_minus_3 - Hsh) + 1e-12
    closer_plus  = abs(H_plus_4  - Hsh) <= abs(H_plus_3  - Hsh) + 1e-12
    err = max(abs(Hexact - Hsh), abs(H_minus_4 - Hsh), abs(H_plus_4 - Hsh))
    return (err < 1e-3) and closer_minus and closer_plus, \
           f"err@1={abs(Hexact-Hsh):.2e}, err@1-1e-4={abs(H_minus_4-Hsh):.2e}, monotone={closer_minus and closer_plus}"

# C: uniform/delta entropy bounds
def _audit_uniform_delta():
    N = 24
    u = np.full(N, 1/N)
    d = np.zeros(N); d[3] = 1
    errs = []
    for q in [0.5, 1.0, 2.0, 5.0]:
        Hu = normalized_renyi_entropy(u, q)
        Hd = normalized_renyi_entropy(d, q)
        errs.append(abs(Hu - 1.0))
        errs.append(abs(Hd - 0.0))
    m = max(errs)
    return m < 1e-9, f"max|err|={m:.2e}"

# D: complexity bounds (audit FINDING-08 fix: also enforce upper bound C_q in [0, 1])
def _audit_complexity_bounds():
    N = 24
    u = np.full(N, 1/N)
    d = np.zeros(N); d[5] = 1
    rng = np.random.default_rng(3)
    bad = []
    for q in [0.5, 1.0, 2.0, 4.0]:
        Cu = statistical_complexity_from_probs(u, q)
        Cd = statistical_complexity_from_probs(d, q)
        for _ in range(200):
            p = rng.dirichlet(np.ones(N))
            Cp = statistical_complexity_from_probs(p, q)
            if Cp < -1e-10:
                bad.append(("lower", q, Cp))
            if Cp > 1.0 + 1e-9:
                bad.append(("upper", q, Cp))
        if abs(Cu) > 1e-9 or abs(Cd) > 1e-9:
            bad.append(("endpoint", q, Cu, Cd))
    return not bad, f"violations={len(bad)} (lower/upper/endpoint)"

# E: cross-check vs ordpy
def _audit_ordpy_xcheck():
    rng = np.random.default_rng(4)
    x = rng.standard_normal(4000)
    dx, q = 4, 2.0
    H_ours, C_ours, p = renyi_point_from_series(x, q=q, dx=dx, tau=1, rng=rng)
    H_o, C_o = _ordpy_renyi_complexity_entropy(x, alpha=q, dx=dx)
    H_o_probs = float(_ordpy_renyi_entropy(p, alpha=q, dx=dx, probs=True))
    H_err = abs(H_ours - H_o)
    Hp_err = abs(H_ours - H_o_probs)
    return (H_err < 1e-6) and (abs(C_ours - C_o) < 1e-3), \
           f"H_err={H_err:.2e}, H_probs_err={Hp_err:.2e}, C_err={abs(C_ours-C_o):.2e}"

# F: ordinal pattern bins == dx!
def _audit_pattern_count():
    for dx in [3, 4, 5]:
        x = np.random.default_rng(0).standard_normal(2000)
        _pats, p = ordinal_distribution(x, dx=dx)
        if len(p) != math.factorial(dx):
            return False, f"dx={dx} returned {len(p)} bins"
    return True, "dx in {3,4,5}"

# G: reproducibility
def _audit_reproducibility():
    cfg = SynthConfig(n_layers=2, n_heads=2, n_tokens=256, seed=7)
    r1 = synthesize_run(cfg)
    r2 = synthesize_run(SynthConfig(n_layers=2, n_heads=2, n_tokens=256, seed=7))
    diff = float(np.max(np.abs(r1["activations"] - r2["activations"])))
    return diff == 0.0, f"max_diff={diff}"

# H: regime separability
def _audit_regime_separability():
    cfg = SynthConfig(n_layers=2, n_heads=8, n_tokens=512, seed=11,
                      layer_regimes=["focused", "diffuse"],
                      head_regimes=[["focused"]*8, ["diffuse"]*8])
    run = synthesize_run(cfg)
    df = feature_table(run, q=2.0, dx=4)
    h_foc = df.loc[df.regime == "focused", "H_act"].mean()
    h_dif = df.loc[df.regime == "diffuse", "H_act"].mean()
    return h_dif - h_foc > 0.05, f"H_diffuse - H_focused = {h_dif-h_foc:+.3f}"

# I: intervention monotonicity
def _audit_intervention_monotonic():
    base = dict(n_layers=4, n_heads=2, n_tokens=384, seed=21,
                layer_regimes=["focused"]*4,
                head_regimes=[["focused"]*2 for _ in range(4)])
    Hs = []
    for s in [0.0, 0.25, 0.5, 0.75, 1.0]:
        cfg = SynthConfig(**base,
                          intervention_strength=s,
                          intervention_layer=1, intervention_head=0,
                          intervention_target_regime="diffuse")
        run = synthesize_run(cfg)
        H, C = renyi_point_from_attention(run["attention"][1, 0], q=2.0)
        Hs.append(H)
    diffs = np.diff(Hs)
    monotonic = bool(np.all(diffs > -1e-3))
    span = (Hs[-1] - Hs[0]) > 0.1
    return monotonic and span, f"H_attn at s=[0..1]={[round(v,3) for v in Hs]}"

# J: bound curves enclose all measured points
def _audit_bounds_enclose():
    q, dx = 2.0, 4
    hc_max = maximum_renyi_complexity_entropy(dx=dx, m=80, q=q)
    hc_min = minimum_renyi_complexity_entropy(dx=dx, size=80, q=q)
    h_grid = np.linspace(0, 1, 200)
    h_max_sorted = hc_max[np.argsort(hc_max[:,0])]
    h_min_sorted = hc_min[np.argsort(hc_min[:,0])]
    Cmax_of_H = np.interp(h_grid, h_max_sorted[:,0], h_max_sorted[:,1])
    Cmin_of_H = np.interp(h_grid, h_min_sorted[:,0], h_min_sorted[:,1])
    cfg = SynthConfig(n_layers=4, n_heads=4, n_tokens=512, seed=37)
    run = synthesize_run(cfg)
    df = feature_table(run, q=q, dx=dx)
    bad = 0
    # Audit FINDING-07: 0.02 is an empirically chosen slack to absorb
    # numerical error from np.interp + bound-curve sampling at m=80. It is
    # NOT theoretically motivated; treat audit J as a smoke test, not a
    # proof. Tightening (e.g. to 0.005) requires re-sampling the bounds at
    # higher m and rejustifying.
    tol = 0.02
    for _, r in df.iterrows():
        H, C = r.H_act, r.C_act
        Cmx = float(np.interp(H, h_grid, Cmax_of_H))
        Cmn = float(np.interp(H, h_grid, Cmin_of_H))
        if C > Cmx + tol or C < Cmn - tol:
            bad += 1
    return bad == 0, f"violations={bad}/{len(df)}"

# K: input validation
def _audit_input_validation():
    p = np.array([0.5, 0.5])
    try:
        renyi_entropy_from_probs(p, q=0.0); return False, "q=0 did not raise"
    except ValueError:
        pass
    try:
        renyi_entropy_from_probs(p, q=-1.0); return False, "q<0 did not raise"
    except ValueError:
        return True, "raised on q=0 and q<0"


# L: q != 1 continuity (audit FINDING-11 — verifies the general formula
#    converges to Shannon as |q - 1| -> 0, beyond just the audit-B point check).
def _audit_q_continuity():
    rng = np.random.default_rng(101)
    p = rng.dirichlet(np.ones(24))
    Hsh = float(-np.sum(p[p>0]*np.log(p[p>0])))
    qs = [0.99, 0.999, 0.9999, 1.0001, 1.001, 1.01]
    errs = []
    for q in qs:
        H = renyi_entropy_from_probs(p, q=q)
        errs.append(abs(H - Hsh))
    # Closer-to-1 must give smaller error than farther-from-1, on each side.
    left_mono  = errs[0] >= errs[1] >= errs[2] - 1e-12
    right_mono = errs[5] >= errs[4] >= errs[3] - 1e-12
    max_err_at_eps3 = max(errs[2], errs[3])  # |q-1| ~ 1e-4
    return left_mono and right_mono and max_err_at_eps3 < 1e-3, \
           f"errs={[f'{e:.1e}' for e in errs]}, monotone_L={left_mono}, monotone_R={right_mono}"

_AUDITS_LIST = [
    ("A: ordinal distribution is on simplex",   _audit_simplex),
    ("B: Renyi q->1 matches Shannon",            _audit_shannon_limit),
    ("C: uniform=>H=1, delta=>H=0",              _audit_uniform_delta),
    ("D: complexity nonneg; uniform/delta=>0",   _audit_complexity_bounds),
    ("E: (H,C) match ordpy",                     _audit_ordpy_xcheck),
    ("F: ordinal bins == dx!",                   _audit_pattern_count),
    ("G: reproducible from seed",                _audit_reproducibility),
    ("H: focused vs diffuse separable",          _audit_regime_separability),
    ("I: intervention strength monotonic",       _audit_intervention_monotonic),
    ("J: all (H,C) inside bound curves",         _audit_bounds_enclose),
    ("K: rejects q<=0",                          _audit_input_validation),
    ("L: q!=1 continuity (general formula)",     _audit_q_continuity),
]
for name, fn in _AUDITS_LIST:
    _run_audit(name, fn)
print()
print(f"Audits run: {len(AUDITS.entries)}; passed: {sum(e[1] for e in AUDITS.entries)}")


## 8. Visualizing the Rényi entropy–complexity plane

Three views — kept restrained per data-viz best practices: 2D only, direct
labelling where practical, color reinforced by marker shape (color is not the
sole encoding), no chartjunk.


In [ ]:
def _draw_bounds(ax, q, dx):
    try:
        hmax = maximum_renyi_complexity_entropy(dx=dx, m=120, q=q)
        hmin = minimum_renyi_complexity_entropy(dx=dx, size=120, q=q)
        smax = hmax[np.argsort(hmax[:,0])]
        smin = hmin[np.argsort(hmin[:,0])]
        ax.plot(smax[:,0], smax[:,1], color="#202020", lw=1.0, zorder=0)
        ax.plot(smin[:,0], smin[:,1], color="#202020", lw=1.0, zorder=0)
        ax.fill(np.r_[smax[:,0], smin[::-1,0]],
                np.r_[smax[:,1], smin[::-1,1]],
                color="#f0f0f0", alpha=0.6, zorder=-1)
    except (ValueError, ArithmeticError, FloatingPointError) as e:
        # Audit FINDING-09: narrowed from bare `except Exception` so
        # programmer errors propagate instead of being swallowed.
        print("Bound curves skipped:", e)

def plot_plane(df, q, dx, ax=None, point_kind="act", title=None,
               show_bounds=True, show_white_noise=True, annotate=False):
    # Audit FINDING-04: the bound curves drawn by `_draw_bounds(ax, q, dx)`
    # are computed for N = dx! (e.g. dx=4 -> N=24). They are correct for the
    # `point_kind="act"` panel where H_act is normalized by log(dx!). They are
    # NOT geometrically correct for `point_kind="attn"` because H_attn is
    # normalized by log(n_tokens) (see `renyi_point_from_attention`), so the
    # attention points live on a different simplex than the envelope. Caller
    # may pass `show_bounds=False` to suppress the misleading envelope on the
    # attention panel; that is the recommended path until matched bounds at
    # N = n_tokens are added.
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.5, 5.2))
    if show_bounds:
        _draw_bounds(ax, q, dx)
    Hcol = "H_attn" if point_kind == "attn" else "H_act"
    Ccol = "C_attn" if point_kind == "attn" else "C_act"
    for regime, sub in df.groupby("regime"):
        base = regime.split("(")[0]
        color = REGIME_COLORS.get(base, "#444")
        marker = REGIME_MARKERS.get(base, "v")
        ax.scatter(sub[Hcol], sub[Ccol], s=70, c=color, marker=marker,
                   edgecolor="white", linewidth=0.8, label=regime, alpha=0.9)
        if annotate:
            for _, r in sub.iterrows():
                ax.annotate(f"L{r.layer}H{r.head}", (r[Hcol], r[Ccol]),
                            fontsize=7, xytext=(3, 3), textcoords="offset points")
    if show_white_noise:
        rng = np.random.default_rng(0)
        x = rng.standard_normal(8000)
        H, C, _ = renyi_point_from_series(x, q=q, dx=dx, rng=rng)
        ax.scatter([H], [C], marker="*", s=120, color="#7f7f7f",
                   edgecolor="black", linewidth=0.5, label="white noise (ref)", zorder=5)
    ax.set_xlim(-0.02, 1.05)
    ax.set_xlabel(r"Normalized Rényi entropy $\mathcal{H}_q$")
    ax.set_ylabel(r"Statistical complexity $C_q$")
    ax.set_title(title or f"{point_kind.title()} plane  q={q}, dx={dx}")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0)
    return ax

def plot_layer_heatmap(df, value="H_act", ax=None, title=None, vmin=0, vmax=1):
    pivot = df.pivot(index="layer", columns="head", values=value)
    if ax is None:
        fig, ax = plt.subplots(figsize=(0.8*pivot.shape[1]+1.5, 0.6*pivot.shape[0]+1.0))
    sns.heatmap(pivot, ax=ax, cmap="viridis", vmin=vmin, vmax=vmax,
                cbar_kws=dict(label=value), annot=True, fmt=".2f",
                linewidths=0.4, linecolor="white")
    ax.set_title(title or f"{value} by layer x head")
    return ax

def plot_intervention_trajectory(layer, head, q=2.0, dx=4, base_seed=99,
                                 target_regime="diffuse", strengths=None, ax=None):
    if strengths is None:
        strengths = np.linspace(0, 1, 9)
    pts = []
    for s in strengths:
        cfg = SynthConfig(n_layers=4, n_heads=2, n_tokens=512, seed=base_seed,
                          layer_regimes=["focused"]*4,
                          head_regimes=[["focused"]*2 for _ in range(4)],
                          intervention_strength=float(s),
                          intervention_layer=layer, intervention_head=head,
                          intervention_target_regime=target_regime)
        run = synthesize_run(cfg)
        H, C = renyi_point_from_attention(run["attention"][layer, head], q=q)
        pts.append((float(s), H, C))
    pts = np.array(pts)
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.6, 5.0))
    _draw_bounds(ax, q, dx)
    sc = ax.scatter(pts[:,1], pts[:,2], c=pts[:,0], cmap="plasma", s=70,
                    edgecolor="white", linewidth=0.6, zorder=4)
    ax.plot(pts[:,1], pts[:,2], color="#8c564b", lw=1, zorder=3)
    cb = plt.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label("Intervention strength alpha")
    for s, H, C in pts[::2]:
        ax.annotate(f"a={s:.2f}", (H, C), fontsize=8, xytext=(4,4), textcoords="offset points")
    ax.set_xlim(-0.02, 1.05)
    ax.set_xlabel(r"Normalized Rényi entropy $\mathcal{H}_q$")
    ax.set_ylabel(r"Statistical complexity $C_q$")
    ax.set_title(f"Causal-patch trajectory  L{layer}H{head} -> {target_regime}  (q={q})")
    return ax

print("Visualization helpers loaded.")


## 9. Static demonstration (works without widgets)

The cells below use a fixed scenario so the notebook produces meaningful output
even when ipywidgets is unavailable (e.g. nbviewer, GitHub preview).


In [ ]:
DEFAULT_CFG = SynthConfig(
    n_layers=6, n_heads=4, n_tokens=512, d_feat=32, seed=0,
    layer_regimes=["focused", "induction", "diffuse", "drift", "collapse", "induction"],
)
run = synthesize_run(DEFAULT_CFG)
df = feature_table(run, q=2.0, dx=4, tau=1)
df.head(10)


In [ ]:
# 9.1 Activation plane
fig, ax = plt.subplots(figsize=(7.0, 5.4))
plot_plane(df, q=2.0, dx=4, ax=ax, point_kind="act", annotate=False,
           title=r"Activation Rényi plane  ($q=2$, $d_x=4$)")
fig.tight_layout(); plt.show()


In [ ]:
# 9.2 Attention plane
fig, ax = plt.subplots(figsize=(7.0, 5.4))
plot_plane(df, q=2.0, dx=4, ax=ax, point_kind="attn",
           title=r"Attention Rényi plane  ($q=2$)")
fig.tight_layout(); plt.show()


In [ ]:
# 9.3 Heatmaps of normalized entropy across layers and heads
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
plot_layer_heatmap(df, value="H_act",  ax=axes[0], title=r"$\mathcal{H}_q$ activation")
plot_layer_heatmap(df, value="H_attn", ax=axes[1], title=r"$\mathcal{H}_q$ attention")
fig.tight_layout(); plt.show()


In [ ]:
# 9.4 Intervention trajectory: a focused head being patched toward diffuse
plot_intervention_trajectory(layer=1, head=0, q=2.0, dx=4,
                             target_regime="diffuse",
                             strengths=np.linspace(0, 1, 9))
plt.tight_layout(); plt.show()


## 10. Interactive controls (Colab-friendly)

Sliders for Rényi order $q$, embedding dimension $d_x$, delay $\tau$,
target layer/head, scenario, seed and intervention strength.
If `ipywidgets` is unavailable the cell prints a hint pointing at §9 for static output.


In [ ]:
def _interactive_panel():
    try:
        import ipywidgets as W
        from IPython.display import display, clear_output
    except Exception:
        print("ipywidgets unavailable -- see Section 9 for static output.")
        return

    out = W.Output()

    q_w = W.FloatSlider(value=2.0, min=0.1, max=7.0, step=0.1, description="q",
                        continuous_update=False)
    dx_w = W.IntSlider(value=4, min=3, max=5, step=1, description="dx", continuous_update=False)
    tau_w = W.IntSlider(value=1, min=1, max=4, step=1, description="tau", continuous_update=False)
    seed_w = W.IntSlider(value=0, min=0, max=99, step=1, description="seed", continuous_update=False)
    n_tok_w = W.IntSlider(value=512, min=128, max=1024, step=64, description="tokens",
                          continuous_update=False)
    layer_w = W.IntSlider(value=1, min=0, max=5, step=1, description="L (target)",
                          continuous_update=False)
    head_w = W.IntSlider(value=0, min=0, max=3, step=1, description="H (target)",
                         continuous_update=False)
    strength_w = W.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description="alpha",
                               continuous_update=False)
    target_w = W.Dropdown(options=REGIMES, value="diffuse", description="patch->")
    scenario_w = W.Dropdown(
        options=[
            ("Default mix",    "default"),
            ("All focused",    "focused"),
            ("All diffuse",    "diffuse"),
            ("Half/half",      "half"),
            ("Induction-rich", "induction"),
        ],
        value="default", description="scenario")

    def _build_cfg():
        s = scenario_w.value
        if s == "default":
            layer_regimes = ["focused", "induction", "diffuse", "drift", "collapse", "induction"]
        elif s == "focused":
            layer_regimes = ["focused"]*6
        elif s == "diffuse":
            layer_regimes = ["diffuse"]*6
        elif s == "half":
            layer_regimes = ["focused"]*3 + ["diffuse"]*3
        elif s == "induction":
            layer_regimes = ["induction"]*6
        else:
            layer_regimes = ["focused"]*6
        cfg = SynthConfig(
            n_layers=6, n_heads=4, n_tokens=int(n_tok_w.value), d_feat=32,
            seed=int(seed_w.value), layer_regimes=layer_regimes,
            intervention_strength=float(strength_w.value),
            intervention_layer=int(layer_w.value),
            intervention_head=int(head_w.value),
            intervention_target_regime=target_w.value,
        )
        return cfg

    def _refresh(*_):
        with out:
            clear_output(wait=True)
            try:
                cfg = _build_cfg()
                run = synthesize_run(cfg)
                df = feature_table(run, q=q_w.value, dx=dx_w.value, tau=tau_w.value)
                fig, axes = plt.subplots(1, 2, figsize=(13, 5))
                plot_plane(df, q=q_w.value, dx=dx_w.value, ax=axes[0], point_kind="act",
                           title=f"Activation plane  q={q_w.value}  dx={dx_w.value}")
                plot_plane(df, q=q_w.value, dx=dx_w.value, ax=axes[1], point_kind="attn",
                           title=f"Attention plane  q={q_w.value}")
                fig.tight_layout(); plt.show()
            except Exception as e:
                print("Error:", e)

    for w in [q_w, dx_w, tau_w, seed_w, n_tok_w, layer_w, head_w, strength_w, target_w, scenario_w]:
        w.observe(_refresh, names="value")
    btn = W.Button(description="Refresh"); btn.on_click(_refresh)
    controls = W.VBox([
        W.HBox([q_w, dx_w, tau_w]),
        W.HBox([scenario_w, n_tok_w, seed_w]),
        W.HBox([layer_w, head_w, strength_w, target_w]),
        btn,
    ])
    display(W.VBox([controls, out]))
    _refresh()

_interactive_panel()


## 11. Final audit summary

We re-print the audit log and assert that everything passed. Use this as a
sanity check after editing any of the math or data-generation cells.


In [ ]:
audit_df = AUDITS.to_dataframe()
try:
    from IPython.display import display as _display
    _display(audit_df)
except Exception:
    print(audit_df.to_string())
print()
print("All passed?", AUDITS.all_passed())
assert AUDITS.all_passed(), "Some audits failed - fix them before relying on the plane geometry."


## 12. Discussion and limitations

**What the plane shows here**

- *Focused* heads cluster at low $\mathcal{H}_q$ with low $C_q$ (delta-like behaviour).
- *Diffuse* heads cluster at high $\mathcal{H}_q$ with low $C_q$ (uniform-like behaviour).
- *Induction* and *drift* regimes sit in the high-complexity belt — they have intermediate entropy *and* far-from-uniform structure.
- *Collapse* heads, due to non-stationarity, often appear in mid-plane for activations but at high $\mathcal{H}_q$ for attention (because the second half of the sequence is uniform).
- *Causal-patch trajectories* sweep monotonically from the source-regime cluster toward the target-regime cluster as $\alpha$ grows.

**What this notebook does *not* prove**

- Real transformer activations are *not* the simple processes used here. The fact that *these* synthetic regimes are separable is necessary but not sufficient for the plane to be informative on real models.
- Bandt–Pompe entropy throws away magnitude information; pre-residual-stream traces with skewed magnitudes may behave very differently from these caricatures.
- $C_q$ depends on the embedding dimension and on $q$. Tuning $q$ to maximize separation is permissible during exploratory analysis but must not be re-used for hypothesis tests on the same data.
- The bound curves are computed only for $q \neq 1$ via the closed form from the source paper; near $q = 1$ we delegate to `ordpy`'s Shannon-case helpers.

**Reproducibility**

- All randomness flows from a single seed in `SynthConfig`. Audit G enforces bit-identical outputs from identical seeds.
- Runtime: end-to-end execution on a Colab CPU is well under a minute for the default scenario.

**Where to go next**

- Replace `synthesize_run` with a real activations cache from a small open-weights transformer; the rest of the pipeline does not change.
- Sweep $q \in (0.2, 5)$ to look for the order that best separates regimes of interest.
- Add Tsallis or Sharma–Mittal entropy variants — the same plane construction generalizes.
